In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Input,GlobalAveragePooling2D, Dense,Dropout,BatchNormalization
from tensorflow.keras.models import Model
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping

img_height,img_width=224,224
num_class=2
num_folds=5
epochs=50
batch_size=32

2025-06-17 04:10:30.840379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750133431.040222      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750133431.100914      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import numpy as np
import tensorflow as tf

# Define parameters
img_height, img_width = 224, 224  # Set your desired image height and width
batch_size = 32  # Set your desired batch size

# Load training dataset with the correct image size
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    '/kaggle/input/aug2splitpcos/splitAug2PreprocessPcos2000/train',
    image_size=(img_height, img_width),  # Ensure images are resized to 224x224
    batch_size=batch_size
)

# Load testing dataset with the correct image size
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    '/kaggle/input/aug2splitpcos/splitAug2PreprocessPcos2000/val',
    image_size=(img_height, img_width),  # Ensure images are resized to 224x224
    batch_size=batch_size
)

# Extracting x_train, y_train
x_train = []
y_train = []
for images, labels in train_dataset:
    x_train.extend(images.numpy())
    y_train.extend(labels.numpy())

# Extracting x_test, y_test
x_test = []
y_test = []
for images, labels in test_dataset:
    x_test.extend(images.numpy())
    y_test.extend(labels.numpy())

# Convert lists to numpy arrays
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")


Found 3200 files belonging to 2 classes.


I0000 00:00:1750133449.635368      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1750133449.636075      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 800 files belonging to 2 classes.
x_train shape: (3200, 224, 224, 3), y_train shape: (3200,)
x_test shape: (800, 224, 224, 3), y_test shape: (800,)


In [3]:
skf=StratifiedKFold(n_splits=num_folds, shuffle=True,random_state=42)
fold_accuracies=[]
num_classes=2

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import NASNetMobile
from tensorflow.keras.layers import (
    Input, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization,
    Conv1D, Reshape, GlobalMaxPooling1D
)
from tensorflow.keras.models import Model
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.callbacks import EarlyStopping

# Define parameters
img_height, img_width = 224, 224
num_classes = len(set(y_train))  # Get the number of classes dynamically
num_folds = 5
epochs = 50
batch_size = 32

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

# Store accuracy for each fold
fold_accuracies = []

for train_index, val_index in skf.split(x_train, y_train):
    x_train_fold, x_val_fold = x_train[train_index], x_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # Define the base model
    base_model = NASNetMobile(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))

    # Input layer
    input_layer = Input(shape=(img_height, img_width, 3))

    # Feature extraction from base model
    base_model_output = base_model(input_layer, training=False)

    # Global Average Pooling
    pooled_output = GlobalAveragePooling2D()(base_model_output)

    # Dense layers
    x = Dense(1024, activation='relu')(pooled_output)
    x = Dropout(0.7)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)

    # Reshape for Conv1D: 256 = 16 x 16
    x = Reshape((16, 16))(x)

    # Conv1D layer
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)

    # Pooling back to 1D
    x = GlobalMaxPooling1D()(x)

    # Output layer
    output_layer = Dense(num_classes, activation='softmax')(x)

    # Build model
    model = Model(inputs=input_layer, outputs=output_layer)

    # Compile model
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Early stopping
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    # Train
    history = model.fit(x_train_fold, y_train_fold,
                        validation_data=(x_val_fold, y_val_fold),
                        epochs=epochs,
                        batch_size=batch_size,
                        verbose=1,
                        callbacks=[early_stopping])

    # Evaluate
    val_loss, val_acc = model.evaluate(x_val_fold, y_val_fold)
    fold_accuracies.append(val_acc)
    print(f"Fold Validation Accuracy: {val_acc:.4f}")

# Final average accuracy
average_accuracy = np.mean(fold_accuracies)
print(f"Average Validation Accuracy across {num_folds} folds: {average_accuracy:.4f}")


19993432/19993432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50


I0000 00:00:1750133632.024334     100 service.cc:148] XLA service 0x7999d8003600 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750133632.025273     100 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1750133632.025302     100 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1750133646.601932     100 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1750133657.012414     100 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1750133657.171288     100 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1750133661.177530     100 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. Th

80/80 ━━━━━━━━━━━━━━━━━━━━ 264s 418ms/step - accuracy: 0.5267 - loss: 0.7958 - val_accuracy: 0.4922 - val_loss: 0.6999
Epoch 2/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 219ms/step - accuracy: 0.8051 - loss: 0.5176 - val_accuracy: 0.5031 - val_loss: 0.7422
Epoch 3/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 223ms/step - accuracy: 0.9836 - loss: 0.1934 - val_accuracy: 0.5156 - val_loss: 0.7715
Epoch 4/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 222ms/step - accuracy: 0.9909 - loss: 0.0690 - val_accuracy: 0.6469 - val_loss: 0.5706
Epoch 5/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 222ms/step - accuracy: 0.9977 - loss: 0.0287 - val_accuracy: 0.7891 - val_loss: 0.4437
Epoch 6/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 223ms/step - accuracy: 0.9979 - loss: 0.0232 - val_accuracy: 0.8328 - val_loss: 0.3634
Epoch 7/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 227ms/step - accuracy: 0.9935 - loss: 0.0295 - val_accuracy: 0.8375 - val_loss: 0.3431
Epoch 8/50
80/80 ━━━━━━━━━━━━━━━━━━━━ 18s 224ms/step - accuracy: 0.9982 - loss: 0.0127 - val_accuracy: 0.84

In [5]:
model.save('/kaggle/working/newnansnet.h5')

In [6]:
from sklearn.metrics import classification_report

# Make predictions on the validation set
y_val_pred_prob = model.predict(x_val_fold)  # Get predicted probabilities
y_val_pred = np.argmax(y_val_pred_prob, axis=1)  # Convert probabilities to class labels
    # Generate classification report for the validation set
report = classification_report(y_val_fold, y_val_pred, target_names=['normal', 'pcos'])  # Adjust class names as needed
print("Classification Report for the Validation Set:")
print(report)
# Calculate and print the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"Average Validation Accuracy across {num_folds} folds: {average_accuracy:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 15s 56ms/step
Classification Report for the Validation Set:
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       320
        pcos       1.00      1.00      1.00       320

    accuracy                           1.00       640
   macro avg       1.00      1.00      1.00       640
weighted avg       1.00      1.00      1.00       640

Average Validation Accuracy across 5 folds: 0.9628
